In [2]:
import os
import pandas as pd

In [3]:
import numpy as np
from sklearn.metrics import accuracy_score, classification_report, balanced_accuracy_score
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier
## PCA
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from imblearn.over_sampling import SMOTE

## svm
from sklearn.svm import SVC
import pandas as pd


In [4]:
ML_CLASSIFIER = ['MLP', 'GaussianNB', "Adaboost", "KNN", "RFClassifier", "SVM_linear", "SVM_sigmoid", "SVM_RBF", "XGBoost"] # "ELM"

def classify(X_train, y_train, X_test, y_test, classifier, search_type='grid'):
    """
    Classify the data using a Random Forest Classifier
    :param X_train: Training data
    :param y_train: Training labels
    :param X_test: Testing data
    :param y_test: Testing labels
    :param search_type: Type of search to perform
    :return: Accuracy of the classifier
    """
    # Create a Random Forest Classifier
    #clf = RandomForestClassifier()

    if classifier == "MLP":
        # clf = MLPClassifier(hidden_layer_sizes=(100,100, 50), max_iter=1000) ### uncomment if the performance is not good with below hyperparameter
        clf = MLPClassifier(hidden_layer_sizes=(100,100, 50), max_iter=1000)
    elif classifier == "GaussianNB": ### {'priors': None, 'var_smoothing': 1e-05}
        clf = GaussianNB(priors=None, var_smoothing= 1e-05) ### Often, the default parameters work well for many datasets.
    elif classifier == "Adaboost":
        dtc = DecisionTreeClassifier(ccp_alpha=0.0, class_weight='balanced', criterion='entropy', max_depth=10, max_features=None, max_leaf_nodes=None, min_impurity_decrease=0.0, min_samples_leaf=2, min_samples_split=2, splitter='best')
        clf = AdaBoostClassifier(estimator=dtc, n_estimators=50, random_state=0, learning_rate=1) ###{'learning_rate': 1, 'n_estimators': 200}
        
    elif classifier == "KNN":
        clf = KNeighborsClassifier(algorithm='auto', leaf_size=10, metric='euclidean', n_jobs=-1,  n_neighbors=1, p=1, weights='uniform')  
    elif classifier == "RFClassifier": ### 
        clf = RandomForestClassifier(bootstrap = False, criterion = 'entropy', max_depth = None, max_features = 'sqrt', min_samples_leaf = 1, min_samples_split = 2, n_estimators = 100, oob_score = False, random_state = 42)
    elif classifier == "SVM_linear": ##
        clf = SVC(kernel='linear')
    elif classifier == "SVM_sigmoid": ###
        clf = SVC(kernel='sigmoid')
    elif classifier == "SVM_RBF": 
        clf = SVC(C=10, cache_size = 200, class_weight = None, gamma = 'scale', kernel = 'rbf', max_iter = -1, probability = True, shrinking = True, tol = 0.001)
    elif classifier == "XGBoost": #'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 300, 'subsample': 0.7
        clf = XGBClassifier(n_estimators=50, learning_rate=0.1, max_depth=3,subsample=0.7)
    else:
        print("Invalid classifier")
        return None
        # Fit the model
    clf.fit(X_train, y_train)

    # Predict the test data
    y_pred = clf.predict(X_test)

    # Calculate the accuracy
    accuracy = balanced_accuracy_score(y_test, y_pred)
    
    return accuracy


In [5]:
data = pd.read_csv('BT-large-4c-dataset_results_finetune_ALL_Models_v3.csv')
data.head(15)

,Model,XGBoost,MLP,GaussianNB,Adaboost,KNN,RFClassifier,SVM_linear,SVM_sigmoid,SVM_RBF,Average
0,resnet50,0.5965,0.7825,0.4749,0.7625,0.7057,0.7617,0.7853,0.2405,0.7927,0.6558
1,resnet101,0.5939,0.7271,0.3372,0.7118,0.7385,0.7669,0.7360,0.2743,0.7213,0.6230
2,densenet121,0.6896,0.7268,0.4551,0.8125,0.7769,0.7546,0.7316,0.6462,0.7254,0.7021
3,densenet169,0.7157,0.7123,0.4262,0.7687,0.7399,0.7387,0.7165,0.6530,0.7094,0.6867
4,vgg16,0.6540,0.7491,0.5991,0.7606,0.7490,0.7424,0.7742,0.6232,0.7727,0.7138
5,vgg19,0.6529,0.7500,0.5452,0.7463,0.7219,0.7366,0.7575,0.5519,0.7766,0.6932
6,alexnet,0.6817,0.7332,0.5025,0.7249,0.7727,0.7391,0.7711,0.4606,0.7663,0.6836
7,resnext50_32x4d,0.5975,0.6324,0.4111,0.7161,0.7330,0.7459,0.6624,0.2339,0.6993,0.6035
8,resnext101_32x8d,0.5518,0.7184,0.3969,0.7151,0.7458,0.7359,0.6710,0.2278,0.7416,0.6116
9,shufflenet_v2_x1_0,0.6232,0.7372,0.5629,0.7111,0.7866,0.7810,0.7536,0.6803,0.7381,0.7082


In [6]:
## taking row-wise average of all the data 
data['average'] = data.drop("Model", axis=1).mean(axis=1)


In [7]:
data.drop("Model", axis=1).mean(axis=0)

XGBoost         0.650477
MLP             0.748492
GaussianNB      0.493223
Adaboost        0.738469
KNN             0.760715
RFClassifier    0.748038
SVM_linear      0.737908
SVM_sigmoid     0.546750
SVM_RBF         0.747442
Average         0.685732
average         0.685725
dtype: float64

In [7]:
## add coumn-wise average of all the data
#data.loc['average'] = data.drop("Model", axis=1).mean(axis=0)

In [8]:
data

,Model,XGBoost,MLP,GaussianNB,Adaboost,KNN,RFClassifier,SVM_linear,SVM_sigmoid,SVM_RBF,Average,average
0,resnet50,0.5965,0.7825,0.4749,0.7625,0.7057,0.7617,0.7853,0.2405,0.7927,0.6558,0.655810
1,resnet101,0.5939,0.7271,0.3372,0.7118,0.7385,0.7669,0.7360,0.2743,0.7213,0.6230,0.623000
2,densenet121,0.6896,0.7268,0.4551,0.8125,0.7769,0.7546,0.7316,0.6462,0.7254,0.7021,0.702080
3,densenet169,0.7157,0.7123,0.4262,0.7687,0.7399,0.7387,0.7165,0.6530,0.7094,0.6867,0.686710
4,vgg16,0.6540,0.7491,0.5991,0.7606,0.7490,0.7424,0.7742,0.6232,0.7727,0.7138,0.713810
5,vgg19,0.6529,0.7500,0.5452,0.7463,0.7219,0.7366,0.7575,0.5519,0.7766,0.6932,0.693210
6,alexnet,0.6817,0.7332,0.5025,0.7249,0.7727,0.7391,0.7711,0.4606,0.7663,0.6836,0.683570
7,resnext50_32x4d,0.5975,0.6324,0.4111,0.7161,0.7330,0.7459,0.6624,0.2339,0.6993,0.6035,0.603510
8,resnext101_32x8d,0.5518,0.7184,0.3969,0.7151,0.7458,0.7359,0.6710,0.2278,0.7416,0.6116,0.611590
9,shufflenet_v2_x1_0,0.6232,0.7372,0.5629,0.7111,0.7866,0.7810,0.7536,0.6803,0.7381,0.7082,0.708220


In [9]:
## sort the data by average
data = data.sort_values(by='average', ascending=False)
data

,Model,XGBoost,MLP,GaussianNB,Adaboost,KNN,RFClassifier,SVM_linear,SVM_sigmoid,SVM_RBF,Average,average
15,vit_small_patch32_224,0.7167,0.7690,0.5493,0.7767,0.8065,0.7852,0.7608,0.5995,0.7795,0.7270,0.727020
11,mnasnet0_5,0.6114,0.7484,0.6710,0.7058,0.7304,0.7710,0.7454,0.7047,0.7450,0.7148,0.714790
4,vgg16,0.6540,0.7491,0.5991,0.7606,0.7490,0.7424,0.7742,0.6232,0.7727,0.7138,0.713810
24,vit_base_patch32_384,0.6751,0.7699,0.4963,0.7692,0.7535,0.7599,0.7773,0.6445,0.7497,0.7106,0.710600
19,vit_small_patch16_224,0.6697,0.7795,0.5317,0.7425,0.8183,0.7619,0.7360,0.5843,0.7560,0.7089,0.708880
9,shufflenet_v2_x1_0,0.6232,0.7372,0.5629,0.7111,0.7866,0.7810,0.7536,0.6803,0.7381,0.7082,0.708220
20,vit_base_patch16_384,0.6917,0.7399,0.4930,0.7494,0.7819,0.7748,0.7419,0.6177,0.7492,0.7044,0.704390
23,vit_small_patch16_384,0.6330,0.7676,0.5083,0.7667,0.7809,0.7579,0.7481,0.6260,0.7489,0.7042,0.704160
2,densenet121,0.6896,0.7268,0.4551,0.8125,0.7769,0.7546,0.7316,0.6462,0.7254,0.7021,0.702080
13,vit_base_patch32_224,0.6835,0.7572,0.4860,0.7437,0.7922,0.7460,0.7404,0.6303,0.7375,0.7019,0.701870


In [10]:
list(data.head(3)['Model'])

['vit_small_patch32_224', 'mnasnet0_5', 'vgg16']

### Top 3 performance model

In [11]:
top_model_combinations = [
                          ['vit_small_patch32_224', 'mnasnet0_5'],
                          ['vit_small_patch32_224', 'vgg16'],
                          ['mnasnet0_5', 'vgg16'],
                          ['vit_small_patch32_224', 'mnasnet0_5', 'vgg16']
                          ]

Ensemble vgg16 with Top 2 networks ['vit_base_patch32_384', 'vit_small_patch32_224', 'vgg16']

In [12]:



# top_model_combinations = [
#                           ['vit_base_patch32_384', 'vit_small_patch32_224'],
#                           ['vit_base_patch32_384', 'vgg16'],
#                           ['vit_small_patch32_224', 'vgg16'],
#                           ['vit_base_patch32_384', 'vit_small_patch32_224', 'vgg16']
#                           ]

In [12]:
columns = ['Model', "XGBoost", 'MLP', 'GaussianNB', "Adaboost", "KNN", "RFClassifier", "SVM_linear", "SVM_sigmoid", "SVM_RBF", ]

dataframe = pd.DataFrame(columns=columns)
# add 12 rows to the dataframe with zero values 
for model_list in top_model_combinations:
    model_name = ' + '.join(model_list)
    new_row = {'Model': model_name, "XGBoost":0, 'MLP': 0, 'GaussianNB': 0, "Adaboost": 0, "KNN": 0, "RFClassifier": 0, "SVM_linear": 0, "SVM_sigmoid": 0, "SVM_RBF": 0} 
    dataframe.loc[len(dataframe)] = new_row
    
model_versions = ['simple_version', "with_normalization_&_PCA", "with_SMOTE_only", "with_normalization_&_PCA_SMOTE"]

model_version_index = 3

main_path = 'extracted_features_BT-large-4c'
for ml_classifier in ML_CLASSIFIER:
    for model_list in top_model_combinations:
        print('Model List:', model_list)
        ensemble_X_train = []
        ensemble_X_test = []

        for model in model_list:
            sub_dir = os.path.join(main_path, model)
            # Load the data
            X_train = np.load(os.path.join(sub_dir, 'train_data_array_features.npy'))
            y_train = np.load(os.path.join(sub_dir, 'train_data_array_labels.npy'))
            X_test = np.load(os.path.join(sub_dir, 'test_data_array_features.npy'))
            y_test = np.load(os.path.join(sub_dir, 'test_data_array_labels.npy'))


            ## squeeze the dimensions 1 from the features 
            X_train = np.squeeze(X_train, axis=1)
            X_test = np.squeeze(X_test, axis=1)

            ## concatenate the features on axis 1
            ensemble_X_train.append(X_train)
            ensemble_X_test.append(X_test)
            

        X_train = np.concatenate(ensemble_X_train, axis=1)
        y_train = y_train
        X_test = np.concatenate(ensemble_X_test, axis=1)
        y_test = y_test

        # ## apply the standard scaler and PCA
        scaler = StandardScaler()
        X_train_normalized = scaler.fit_transform(X_train)
        X_test_normalized = scaler.transform(X_test)

        # Step 2 & 3: Apply PCA
        n_components = 8000#int(0.45* X_train.shape[1])
        pca = PCA(n_components=n_components)
        X_train = pca.fit_transform(X_train_normalized)
        X_test = pca.transform(X_test_normalized)

        # ## no of example in each class
        print("no of example in each class before SMOTE:", np.bincount(y_train))


        # ## over sample the data (data augmentation)
        smote = SMOTE(sampling_strategy='minority')
        X_resampled, y_resampled = smote.fit_resample(X_train, y_train)

        # ## no of example in each class
        print("no of example in each class after SMOTE:", np.bincount(y_resampled))


        # print("no of features in X_train:", X_train.shape)

        
        # Classify the data
        accuracy = classify(X_train, y_train, X_test, y_test, ml_classifier)
        print('Accuracy:', accuracy)
        model_name = ' + '.join(model_list)

        dataframe.loc[dataframe['Model'] == model_name, ml_classifier] = accuracy

    print(dataframe)
    dataframe.to_csv(f'BT-large-4c-dataset_results_top_three_{model_versions[model_version_index]}.csv', index=False)
    # dataframe.to_csv('BT-large-4c-dataset_results_top_three_normalized_PCA_tuned_parameters.csv', index=False)


Model List: ['vit_small_patch32_224', 'mnasnet0_5']
no of example in each class before SMOTE: [2475 2441 1124 2511]
no of example in each class after SMOTE: [2475 2441 2511 2511]
Accuracy: 0.7536574618096357
Model List: ['vit_small_patch32_224', 'vgg16']


C:\Users\user\AppData\Local\Temp\ipykernel_35056\4055229502.py:75: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.7536574618096357' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  dataframe.loc[dataframe['Model'] == model_name, ml_classifier] = accuracy


no of example in each class before SMOTE: [2475 2441 1124 2511]
no of example in each class after SMOTE: [2475 2441 2511 2511]
Accuracy: 0.7640981198589895
Model List: ['mnasnet0_5', 'vgg16']
no of example in each class before SMOTE: [2475 2441 1124 2511]
no of example in each class after SMOTE: [2475 2441 2511 2511]
Accuracy: 0.7523531139835486
Model List: ['vit_small_patch32_224', 'mnasnet0_5', 'vgg16']
no of example in each class before SMOTE: [2475 2441 1124 2511]
no of example in each class after SMOTE: [2475 2441 2511 2511]
Accuracy: 0.7503143360752057
                                        Model  XGBoost       MLP  GaussianNB  \
0          vit_small_patch32_224 + mnasnet0_5        0  0.753657           0   
1               vit_small_patch32_224 + vgg16        0  0.764098           0   
2                          mnasnet0_5 + vgg16        0  0.752353           0   
3  vit_small_patch32_224 + mnasnet0_5 + vgg16        0  0.750314           0   

   Adaboost  KNN  RFClassifier  SV

C:\Users\user\AppData\Local\Temp\ipykernel_35056\4055229502.py:75: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.2659970063230933' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  dataframe.loc[dataframe['Model'] == model_name, ml_classifier] = accuracy


no of example in each class before SMOTE: [2475 2441 1124 2511]
no of example in each class after SMOTE: [2475 2441 2511 2511]
Accuracy: 0.44489494152537634
Model List: ['mnasnet0_5', 'vgg16']
no of example in each class before SMOTE: [2475 2441 1124 2511]
no of example in each class after SMOTE: [2475 2441 2511 2511]
Accuracy: 0.27198757763975157
Model List: ['vit_small_patch32_224', 'mnasnet0_5', 'vgg16']
no of example in each class before SMOTE: [2475 2441 1124 2511]
no of example in each class after SMOTE: [2475 2441 2511 2511]
Accuracy: 0.27536595601812996
                                        Model  XGBoost       MLP  GaussianNB  \
0          vit_small_patch32_224 + mnasnet0_5        0  0.753657    0.265997   
1               vit_small_patch32_224 + vgg16        0  0.764098    0.444895   
2                          mnasnet0_5 + vgg16        0  0.752353    0.271988   
3  vit_small_patch32_224 + mnasnet0_5 + vgg16        0  0.750314    0.275366   

   Adaboost  KNN  RFClassifier 

...

no of example in each class before SMOTE: [2475 2441 1124 2511]
no of example in each class after SMOTE: [2475 2441 2511 2511]
Accuracy: 0.7309313972357451
Model List: ['mnasnet0_5', 'vgg16']
no of example in each class before SMOTE: [2475 2441 1124 2511]
no of example in each class after SMOTE: [2475 2441 2511 2511]
Accuracy: 0.7139866263779308
Model List: ['vit_small_patch32_224', 'mnasnet0_5', 'vgg16']
no of example in each class before SMOTE: [2475 2441 1124 2511]
no of example in each class after SMOTE: [2475 2441 2511 2511]
Accuracy: 0.7207433831346874
                                        Model  XGBoost       MLP  GaussianNB  \
0          vit_small_patch32_224 + mnasnet0_5        0  0.753657    0.265997   
1               vit_small_patch32_224 + vgg16        0  0.764098    0.444895   
2                          mnasnet0_5 + vgg16        0  0.752353    0.271988   
3  vit_small_patch32_224 + mnasnet0_5 + vgg16        0  0.750314    0.275366   

   Adaboost       KNN  RFClassifie

C:\Users\user\AppData\Local\Temp\ipykernel_35056\4055229502.py:75: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.7755611045828437' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  dataframe.loc[dataframe['Model'] == model_name, ml_classifier] = accuracy


no of example in each class before SMOTE: [2475 2441 1124 2511]
no of example in each class after SMOTE: [2475 2441 2511 2511]
Accuracy: 0.7408666274970623
Model List: ['mnasnet0_5', 'vgg16']
no of example in each class before SMOTE: [2475 2441 1124 2511]
no of example in each class after SMOTE: [2475 2441 2511 2511]
Accuracy: 0.7920710928319623
Model List: ['vit_small_patch32_224', 'mnasnet0_5', 'vgg16']
no of example in each class before SMOTE: [2475 2441 1124 2511]
no of example in each class after SMOTE: [2475 2441 2511 2511]
Accuracy: 0.7718008225616921
                                        Model  XGBoost       MLP  GaussianNB  \
0          vit_small_patch32_224 + mnasnet0_5        0  0.753657    0.265997   
1               vit_small_patch32_224 + vgg16        0  0.764098    0.444895   
2                          mnasnet0_5 + vgg16        0  0.752353    0.271988   
3  vit_small_patch32_224 + mnasnet0_5 + vgg16        0  0.750314    0.275366   

   Adaboost       KNN  RFClassifie

C:\Users\user\AppData\Local\Temp\ipykernel_35056\4055229502.py:75: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.5437037658776789' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  dataframe.loc[dataframe['Model'] == model_name, ml_classifier] = accuracy


no of example in each class before SMOTE: [2475 2441 1124 2511]
no of example in each class after SMOTE: [2475 2441 2511 2511]
Accuracy: 0.5102348777348777
Model List: ['mnasnet0_5', 'vgg16']
no of example in each class before SMOTE: [2475 2441 1124 2511]
no of example in each class after SMOTE: [2475 2441 2511 2511]
Accuracy: 0.46614683005987356
Model List: ['vit_small_patch32_224', 'mnasnet0_5', 'vgg16']
no of example in each class before SMOTE: [2475 2441 1124 2511]
no of example in each class after SMOTE: [2475 2441 2511 2511]
Accuracy: 0.5022793911924346
                                        Model   XGBoost       MLP  GaussianNB  \
0          vit_small_patch32_224 + mnasnet0_5  0.543704  0.753657    0.265997   
1               vit_small_patch32_224 + vgg16  0.510235  0.764098    0.444895   
2                          mnasnet0_5 + vgg16  0.466147  0.752353    0.271988   
3  vit_small_patch32_224 + mnasnet0_5 + vgg16  0.502279  0.750314    0.275366   

   Adaboost       KNN  RFCla

In [18]:
n_components

28396

In [14]:
dataframe

,Model,XGBoost,MLP,GaussianNB,Adaboost,KNN,RFClassifier,SVM_linear,SVM_sigmoid,SVM_RBF
0,vit_base_patch32_384 + vit_small_patch32_224,0.736086,0.762759,0.334131,0.731610,0.771137,0.738784,0.799189,0.683773,0.754797
1,vit_base_patch32_384 + vit_small_patch16_224,0.733586,0.758041,0.333893,0.754932,0.796542,0.748919,0.779797,0.658766,0.754797
2,vit_small_patch32_224 + vit_small_patch16_224,0.740533,0.756554,0.410204,0.764571,0.812421,0.777488,0.719953,0.619690,0.765811
3,vit_base_patch32_384 + vit_small_patch32_224 +...,0.721848,0.767432,0.325231,0.747432,0.802421,0.782297,0.771554,0.665940,0.762297
